# Gold Logistique
Pipeline Olist : indicateurs logistiques à partir de la zone silver.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, round as _round,
    countDistinct, date_format, max as _max,
    collect_set, concat_ws, create_map, lit, coalesce
)
from itertools import chain

spark = SparkSession.builder \
    .appName("olist-gold-logistique") \
    .getOrCreate()



## Chargement des tables silver

In [ ]:
df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_pcnt = spark.read.parquet("../data/silver/product_category_name_translation/")

## Segmentation des commandes par statut
Trois segments : revenu reconnu, en transit, exclu.

In [ ]:
df_orders.groupBy("order_status").count().show()

In [ ]:
# Trois segments selon le statut de la commande
df_orders_delivered = df_orders.filter(col("order_status") == "delivered")

df_orders_in_transit = df_orders.filter(
    col("order_status").isin(["created", "approved", "processing", "invoiced", "shipped"])
)

df_orders_excluded = df_orders.filter(
    col("order_status").isin(["canceled", "unavailable"])
)

## Calcul des délais de livraison

In [ ]:
from pyspark.sql.functions import datediff, col, lit, when, round as _round

# Calcul du délai de livraison en jours pour les commandes livrées
df_delivery_delays = df_orders_delivered \
    .withColumn("delivery_delay_days", 
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .select("order_id", "delivery_delay_days")

# Statistiques des délais de livraison
df_delivery_delays.describe().show()

# Délai moyen de livraison
avg_delivery_delay = df_delivery_delays.agg(_round(avg("delivery_delay_days"), 2).alias("avg_delay_days"))
avg_delivery_delay.show()

## Taux de respect des délais

In [ ]:
# Calcul du respect des délais (livraison avant la date estimée)
df_delivery_performance = df_orders_delivered \
    .withColumn("on_time", 
                when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1)
                .otherwise(0)) \
    .withColumn("delay_days", 
                datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))

# Taux de respect des délais
on_time_rate = df_delivery_performance.agg(
    (_round(avg("on_time") * 100, 2)).alias("on_time_rate_pct")
)
on_time_rate.show()

# Statistiques sur les retards
df_delivery_performance.agg(
    avg("delay_days").alias("avg_delay_days"),
    count(when(col("delay_days") > 0, 1)).alias("late_deliveries"),
    count(when(col("delay_days") <= 0, 1)).alias("on_time_deliveries")
).show()

## Performance par région/état

In [ ]:
# Jointure avec les clients pour obtenir la région/état
df_orders_with_customers = df_orders_delivered.join(
    df_customers, 
    df_orders_delivered["customer_id"] == df_customers["customer_id"],
    "inner"
)

# Calcul des délais par état
df_performance_by_state = df_orders_with_customers \
    .withColumn("delivery_delay_days", 
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .withColumn("on_time", 
                when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1)
                .otherwise(0)) \
    .groupBy("customer_state") \
    .agg(
        count("*").alias("total_orders"),
        _round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay_days"),
        _round(avg("on_time") * 100, 2).alias("on_time_rate_pct"),
        count(when(col("on_time") == 0, 1)).alias("late_orders")
    ) \
    .orderBy("avg_delivery_delay_days")

df_performance_by_state.show(30)

## Coûts de livraison

In [ ]:
# Calcul des coûts de livraison par commande
df_freight_costs = df_items.groupBy("order_id") \
    .agg(
        _round(_sum("freight_value"), 2).alias("total_freight_cost"),
        count("*").alias("item_count"),
        _round(avg("freight_value"), 2).alias("avg_freight_per_item")
    )

# Statistiques des coûts de livraison
df_freight_costs.describe().show()

# Coût moyen de livraison
avg_freight = df_freight_costs.agg(
    _round(avg("total_freight_cost"), 2).alias("avg_freight_cost")
)
avg_freight.show()

## Taux de retards

In [ ]:
# Calcul du taux de retards (livraison après la date estimée)
df_late_deliveries = df_orders_delivered \
    .withColumn("is_late", 
                when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1)
                .otherwise(0)) \
    .withColumn("delay_days", 
                datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))

# Taux global de retards
late_rate = df_late_deliveries.agg(
    (_round(avg("is_late") * 100, 2)).alias("late_rate_pct")
)
late_rate.show()

# Statistiques détaillées sur les retards
df_late_deliveries.agg(
    count("*").alias("total_delivered"),
    count(when(col("is_late") == 1, 1)).alias("late_count"),
    count(when(col("is_late") == 0, 1)).alias("on_time_count"),
    _round(avg("delay_days"), 2).alias("avg_delay_days_when_late"),
    _round(_max("delay_days"), 2).alias("max_delay_days")
).show()

# Distribution des retards par nombre de jours
df_late_deliveries.filter(col("is_late") == 1) \
    .groupBy("delay_days") \
    .count() \
    .orderBy("delay_days") \
    .show(20)

## Performance par vendeur

In [ ]:
# Jointure des commandes avec les items pour obtenir les vendeurs
df_orders_with_sellers = df_orders_delivered.join(
    df_items,
    "order_id",
    "inner"
).join(
    df_sellers,
    "seller_id",
    "inner"
)

# Calcul de la performance par vendeur
df_seller_performance = df_orders_with_sellers \
    .withColumn("delivery_delay_days", 
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .withColumn("on_time", 
                when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1)
                .otherwise(0)) \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(
        count("*").alias("total_orders"),
        _round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay_days"),
        _round(avg("on_time") * 100, 2).alias("on_time_rate_pct"),
        count(when(col("on_time") == 0, 1)).alias("late_orders"),
        _round(_sum("freight_value"), 2).alias("total_freight_value"),
        _round(avg("freight_value"), 2).alias("avg_freight_value")
    ) \
    .orderBy("total_orders", ascending=False)

df_seller_performance.show(30)

## Sauvegarde des résultats dans le niveau Gold

In [ ]:
# Création du répertoire de sortie
import os
os.makedirs("../data/gold/logistique", exist_ok=True)

# Sauvegarde des indicateurs logistiques
df_delivery_delays.write.mode("overwrite").parquet("../data/gold/logistique/delivery_delays/")
df_performance_by_state.write.mode("overwrite").parquet("../data/gold/logistique/performance_by_state/")
df_freight_costs.write.mode("overwrite").parquet("../data/gold/logistique/freight_costs/")
df_seller_performance.write.mode("overwrite").parquet("../data/gold/logistique/seller_performance/")

print("Indicateurs logistiques sauvegardés dans data/gold/logistique/")